# GeoInfraLens AI – P8.2: clean and split the dataset

Downloads RDD2022 again (Colab forgets files when a session ends; about 10 minutes), then cleans it and splits it:
- **test** = every labelled Czech photo (locked; never used for training),
- **validation** = about 10% of each other country, **training** = the rest,
- China drone photos and the United States part are left out,
- duplicate and near-duplicate photos are handled so the test can't be cheated.

**How to open this notebook:** go to colab.research.google.com → **File → Upload notebook** → choose this file from your computer (`ml/notebooks/p8_2_split_rdd2022.ipynb`). Nothing needs uploading to Google Drive.

**How to run:** click ▶ on each grey cell from top to bottom and wait for the green tick before the next. No GPU is needed. Keep this tab open and your laptop awake.

Only the public dataset is downloaded; nothing from your app or computer is sent here.

Total time: about 40–60 minutes. At the end your browser downloads **`p8_2_results.zip`** (small: counts and lists, no photos).

*Generated from `ml/data_prep/*.py` by `ml/notebooks/make_notebooks.py`. Do not edit by hand.*


In [ ]:
%%writefile fetch_rdd2022.py
# 1. Saves the download script (nothing runs yet)
"""P8.1: download RDD2022, check it, unpack it WITHOUT the United States part, and count what is inside.

Runs on Google Colab (the dataset is 13.3 GB). Plan: documentation/04-design/ml-baseline-plan.md.

    python fetch_rdd2022.py --out /content/rdd2022

Steps:
1. Download the official zip from figshare (resumes if interrupted).
2. Check its size and MD5 fingerprint against the values figshare publishes, so a broken or
   changed file is never used.
3. Unpack it. Anything under a "United_States" folder is skipped and never written to disk:
   those photos are Google Street View images, and Google's terms forbid using them to train
   or test AI models (plan section 6). Zips inside the zip are handled the same way.
4. Count images, label files and boxes per country and per damage code, and save the counts
   to inventory.json (used to fill in the dataset card).

The dataset is outside data, so the label files are read with defusedxml (safe against
malicious XML), and every path inside the zip is checked before anything is written.
"""

from __future__ import annotations

import argparse
import hashlib
import json
import re
import shutil
import sys
import tempfile
import time
import urllib.error
import urllib.request
import zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath
from typing import Any

from defusedxml import ElementTree

# Official record: https://figshare.com/articles/dataset/21431547 (CC BY 4.0), checked 2026-10-16.
DATASET_URL = "https://ndownloader.figshare.com/files/38030910"
DATASET_FILE = "RDD2022_released_through_CRDDC2022.zip"
DATASET_SIZE = 13_264_172_619
DATASET_MD5 = "b62bd51d2ffcfaa76c60f234f0cc2bb3"

# Never used, in any form (plan section 6).
EXCLUDED = re.compile(r"united[\s_-]*states", re.IGNORECASE)
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}
CHUNK = 8 * 1024 * 1024
# The download site sometimes refuses or drops big downloads for a while (e.g. "403 Forbidden"
# from busy shared Colab addresses). Wait and try again, continuing the partial file.
RETRY_WAITS_SECONDS = (30, 90, 180, 300)
RETRY_STATUS = {403, 408, 429, 500, 502, 503, 504}
USER_AGENT = "GeoInfraLens-dataset-fetch/1.0 (one-time research download of RDD2022)"


class DatasetError(Exception):
    """A problem that must stop the run (wrong file, unsafe content)."""


# ---------------------------------------------------------------- download and check


def download(url: str, dest: Path, expected_size: int) -> None:
    """Download `url` to `dest`, continuing a partial file, and retrying when the site refuses for a while."""
    if not url.startswith(("https://", "http://127.0.0.1")):  # the official https URL (local address only in tests)
        raise DatasetError(f"Refusing to download from {url!r}")
    for attempt, wait in enumerate((*RETRY_WAITS_SECONDS, None), start=1):
        try:
            _download_once(url, dest, expected_size)
            return
        except urllib.error.HTTPError as error:
            if error.code not in RETRY_STATUS:
                raise DatasetError(f"The download site answered {error.code} {error.reason}.") from error
            problem = f"the download site answered {error.code} {error.reason}"
        except (urllib.error.URLError, ConnectionError, TimeoutError) as error:
            problem = f"the connection failed ({error})"
        if wait is None:
            raise DatasetError(
                f"Gave up after {attempt} tries: {problem}. The site may be limiting downloads for a while. "
                "Wait 15-30 minutes and run this cell again (it continues where it stopped), or in Colab use "
                "Runtime > Disconnect and delete runtime, then run all cells again (you get a new internet address)."
            )
        print(f"  Try {attempt} failed: {problem}. Waiting {wait} seconds, then continuing...")
        time.sleep(wait)


def _download_once(url: str, dest: Path, expected_size: int) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    have = dest.stat().st_size if dest.exists() else 0
    if have == expected_size:
        print(f"Already downloaded: {dest}")
        return
    if have > expected_size:
        raise DatasetError(f"{dest} is larger than expected; delete it and run again.")
    headers = {"User-Agent": USER_AGENT}
    if have:
        headers["Range"] = f"bytes={have}-"
    request = urllib.request.Request(url, headers=headers)  # noqa: S310 (scheme checked in download)
    with urllib.request.urlopen(request, timeout=60) as response:  # noqa: S310 (scheme checked in download)
        if have and response.status != 206:
            have = 0  # server ignored the resume request: start again
        mode = "ab" if have else "wb"
        done, next_report, started = have, have, time.monotonic()
        with dest.open(mode) as handle:
            while chunk := response.read(CHUNK):
                handle.write(chunk)
                done += len(chunk)
                if done >= next_report:
                    speed = (done - have) / max(time.monotonic() - started, 0.001) / 1e6
                    print(f"  {done / expected_size:6.1%}  ({done / 1e9:.2f} of {expected_size / 1e9:.2f} GB, {speed:.0f} MB/s)")
                    next_report = done + expected_size // 20
    if dest.stat().st_size != expected_size:
        raise ConnectionError(f"download stopped early ({dest.stat().st_size} of {expected_size} bytes)")


def check_md5(path: Path, expected: str) -> None:
    digest = hashlib.md5()  # noqa: S324 (file fingerprint published by figshare, not security)
    with path.open("rb") as handle:
        while chunk := handle.read(CHUNK):
            digest.update(chunk)
    if digest.hexdigest() != expected:
        raise DatasetError(
            f"MD5 mismatch for {path.name}: got {digest.hexdigest()}, expected {expected}. "
            "The file is damaged or not the official one; delete it and download again."
        )
    print(f"MD5 OK: {expected}")


# ---------------------------------------------------------------- unpack without the US part


def safe_relative_path(name: str) -> PurePosixPath:
    """The path of a zip entry, refusing anything that could write outside the output folder."""
    path = PurePosixPath(name.replace("\\", "/"))
    if path.is_absolute() or ".." in path.parts or (path.parts and ":" in path.parts[0]):
        raise DatasetError(f"Unsafe path inside the zip: {name!r}")
    return path


def is_excluded(path: PurePosixPath) -> bool:
    return any(EXCLUDED.search(part) for part in path.parts)


def extract(zip_path: Path, out_dir: Path, report: Counter[str], prefix: PurePosixPath | None = None) -> None:
    """Unpack `zip_path` into `out_dir`, skipping excluded parts. Zips inside are unpacked in place."""
    prefix = prefix or PurePosixPath()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            relative = prefix / safe_relative_path(member.filename)
            if is_excluded(relative):
                report["skipped_excluded"] += 1
                continue
            if member.is_dir():
                continue
            if relative.suffix.lower() == ".zip":
                # Unpack the inner zip into a folder named after it (without ".zip").
                with tempfile.TemporaryDirectory() as tmp:
                    inner = Path(tmp) / relative.name
                    with archive.open(member) as source, inner.open("wb") as target:
                        shutil.copyfileobj(source, target, CHUNK)
                    report["inner_zips"] += 1
                    extract(inner, out_dir, report, relative.parent / relative.stem)
                continue
            target = out_dir.joinpath(*relative.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open("wb") as handle:
                shutil.copyfileobj(source, handle, CHUNK)
            report["files_written"] += 1


# ---------------------------------------------------------------- count what is inside


def country_and_split(path: PurePosixPath) -> tuple[str, str]:
    """Country folder and split, from paths like <...>/Japan/train/images/x.jpg."""
    parts = [part.lower() for part in path.parts]
    for index, part in enumerate(parts):
        if part in {"train", "test"} and index > 0:
            return path.parts[index - 1], part
    return "unknown", "unknown"


def read_labels(xml_path: Path) -> tuple[list[str], tuple[int, int] | None]:
    """Box label names and the image size from one Pascal VOC label file."""
    root = ElementTree.parse(xml_path).getroot()
    names = [(obj.findtext("name") or "").strip() for obj in root.iter("object")]
    width, height = root.findtext("size/width"), root.findtext("size/height")
    size = (int(float(width)), int(float(height))) if width and height else None
    return names, size


def build_inventory(out_dir: Path) -> dict[str, Any]:
    countries: dict[str, dict[str, Any]] = defaultdict(
        lambda: {"images": Counter(), "label_files": 0, "boxes": Counter(), "images_without_boxes": 0, "image_sizes": Counter()}
    )
    problems: list[str] = []
    for file in sorted(out_dir.rglob("*")):
        if not file.is_file():
            continue
        relative = PurePosixPath(file.relative_to(out_dir).as_posix())
        if is_excluded(relative):
            raise DatasetError(f"Excluded data found on disk: {relative}. Delete it before continuing.")
        country, split = country_and_split(relative)
        entry = countries[country]
        suffix = file.suffix.lower()
        if suffix in IMAGE_SUFFIXES:
            entry["images"][split] += 1
        elif suffix == ".xml":
            entry["label_files"] += 1
            try:
                names, size = read_labels(file)
            except Exception as error:  # noqa: BLE001 (report every broken file, keep counting)
                problems.append(f"{relative}: {error}")
                continue
            entry["boxes"].update(names)
            if not names:
                entry["images_without_boxes"] += 1
            if size:
                entry["image_sizes"][f"{size[0]}x{size[1]}"] += 1
    result = {
        name: {
            "images": dict(data["images"]),
            "label_files": data["label_files"],
            "images_without_boxes": data["images_without_boxes"],
            "boxes_by_label": dict(sorted(data["boxes"].items())),
            "common_image_sizes": dict(data["image_sizes"].most_common(3)),
        }
        for name, data in sorted(countries.items())
    }
    return {"countries": result, "unreadable_label_files": problems[:50], "unreadable_label_file_count": len(problems)}


def print_summary(inventory: dict[str, Any]) -> None:
    labels = sorted({label for data in inventory["countries"].values() for label in data["boxes_by_label"]})
    print("\nCountry            train img  test img  " + "  ".join(f"{label:>6}" for label in labels))
    for name, data in inventory["countries"].items():
        boxes = "  ".join(f"{data['boxes_by_label'].get(label, 0):>6}" for label in labels)
        print(f"{name:<18} {data['images'].get('train', 0):>9}  {data['images'].get('test', 0):>8}  {boxes}")
    if inventory["unreadable_label_file_count"]:
        print(f"\n{inventory['unreadable_label_file_count']} label files could not be read (listed in inventory.json).")


# ---------------------------------------------------------------- command line


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--out", type=Path, required=True, help="Folder for the zip, the unpacked data and inventory.json")
    parser.add_argument("--keep-zip", action="store_true", help="Keep the 13 GB zip after unpacking (default: delete it to save disk)")
    args = parser.parse_args(argv)

    zip_path = args.out / DATASET_FILE
    data_dir = args.out / "data"
    try:
        print(f"1/4 Downloading {DATASET_FILE} ({DATASET_SIZE / 1e9:.1f} GB)")
        download(DATASET_URL, zip_path, DATASET_SIZE)
        print("2/4 Checking the file fingerprint (MD5)")
        check_md5(zip_path, DATASET_MD5)
        print("3/4 Unpacking (United States part skipped)")
        if data_dir.exists():
            shutil.rmtree(data_dir)  # always unpack fresh, so nothing excluded or stale survives
        report: Counter[str] = Counter()
        extract(zip_path, data_dir, report)
        print(
            f"   files written: {report['files_written']}, inner zips: {report['inner_zips']}, "
            f"US entries skipped: {report['skipped_excluded']}"
        )
        if not args.keep_zip:
            zip_path.unlink()
        print("4/4 Counting images and boxes")
        inventory = build_inventory(data_dir)
        inventory["source"] = {
            "url": DATASET_URL,
            "file": DATASET_FILE,
            "bytes": DATASET_SIZE,
            "md5": DATASET_MD5,
            "licence": "CC BY 4.0 (figshare record); authors' GitHub states CC BY-SA 4.0",
        }
        inventory["unpack_report"] = dict(report)
        inventory["created_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        (args.out / "inventory.json").write_text(json.dumps(inventory, indent=2))
        print_summary(inventory)
        print(f"\nSaved {args.out / 'inventory.json'}")
    except DatasetError as error:
        print(f"STOPPED: {error}", file=sys.stderr)
        return 1
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile split_rdd2022.py
# 2. Saves the clean-and-split script (nothing runs yet)
"""P8.2: clean RDD2022 and split it into training, validation and test sets (test is then locked).

Runs on Google Colab right after fetch_rdd2022.py, on its output folder:

    python split_rdd2022.py --data /content/rdd2022/data --out /content/rdd2022/prepared

Rules (documentation/04-design/ml-baseline-plan.md, sections 2-3; decisions of 2026-10-16):
- Only labelled photos are used. RDD2022's own "test" photos have no labels and are ignored.
- Test set = every labelled Czech photo (a whole country the model never trains on).
- China_Drone is left out of model 0.1 (drone view from above, unlike phone photos).
- RDD2022 codes become the app's 3 classes; every other code is ignored.
- Boxes are clipped to the photo; boxes thinner than 2 pixels are dropped.
- Exact duplicate photos are kept once. Near-duplicates (almost the same photo, e.g. two
  frames a second apart) are grouped and always kept in the same set, so the model is never
  checked on a photo it has practically seen. Any training photo that is a near-duplicate of a
  test photo is removed.
- Validation = about 10% of each training country, picked by group with a fixed seed, so
  re-running gives exactly the same split.
- Big photos (Norway, about 3,650 pixels wide) are shrunk so the longest side is at most
  1,333 pixels (the size the model works at anyway); boxes are scaled with them.

Output (in --out):
- images/<country>/<name>.jpg          prepared photos
- annotations/{train,val,test}.json   boxes in COCO format (the format the evaluation reads)
- splits/{train,val,test}.txt          which original photo went where
- split-manifest.json                  counts, settings and fingerprints; the test set is
                                       "locked" by the SHA-256 of its list and annotations
"""

from __future__ import annotations

import argparse
import hashlib
import json
import random
import sys
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path, PurePosixPath
from typing import Any

import numpy as np
from defusedxml import ElementTree
from PIL import Image

try:  # inside the repository
    from data_prep.fetch_rdd2022 import country_and_split, is_excluded
except ImportError:  # on Colab, both scripts sit in the same folder
    from fetch_rdd2022 import country_and_split, is_excluded  # type: ignore[no-redef]

SPLIT_VERSION = 1
SEED = 2026
TEST_COUNTRY = "Czech"
LEFT_OUT_COUNTRIES = {"China_Drone"}
VAL_FRACTION = 0.10
MAX_SIDE = 1333
MIN_BOX_SIDE = 2.0
HASH_SIZE = 16  # 16 x 16 = 256-bit difference hash
NEAR_DUPLICATE_BITS = 10  # at most 10 of 256 bits different = near-duplicate
JPEG_QUALITY = 92

# App class names in a fixed order: the model's class numbers depend on it. Never reorder.
CATEGORIES = [
    {"id": 1, "name": "pothole"},
    {"id": 2, "name": "surface_cracking"},
    {"id": 3, "name": "pavement_deterioration"},
]
CLASS_FROM_CODE = {"D40": "pothole", "D00": "surface_cracking", "D10": "surface_cracking", "D20": "pavement_deterioration"}
CATEGORY_ID = {category["name"]: category["id"] for category in CATEGORIES}
IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png")


@dataclass
class Box:
    code: str
    xmin: float
    ymin: float
    xmax: float
    ymax: float


@dataclass
class Photo:
    path: Path
    relative: str  # path inside the data folder, with forward slashes
    country: str
    width: int
    height: int
    boxes: list[Box] = field(default_factory=list)
    sha1: str = ""
    dhash: np.ndarray | None = None  # 4 x uint64 (256 bits)
    group: int = -1


# ---------------------------------------------------------------- reading


def read_boxes(xml_path: Path) -> tuple[tuple[int, int] | None, list[Box]]:
    """Image size (if given) and every box in one Pascal VOC label file."""
    root = ElementTree.parse(xml_path).getroot()
    width, height = root.findtext("size/width"), root.findtext("size/height")
    size = (int(float(width)), int(float(height))) if width and height and float(width) > 0 and float(height) > 0 else None
    boxes = []
    for obj in root.iter("object"):
        code = (obj.findtext("name") or "").strip()
        values = [obj.findtext(f"bndbox/{key}") for key in ("xmin", "ymin", "xmax", "ymax")]
        if code and all(values):
            boxes.append(Box(code, *(float(value) for value in values)))  # type: ignore[arg-type]
    return size, boxes


def find_photo(xml_path: Path) -> Path | None:
    """The photo for <...>/train/annotations/xmls/name.xml is <...>/train/images/name.jpg."""
    images = xml_path.parent.parent.parent / "images"
    for suffix in IMAGE_SUFFIXES:
        candidate = images / f"{xml_path.stem}{suffix}"
        if candidate.exists():
            return candidate
    return None


def collect_photos(data_dir: Path, report: Counter[str]) -> list[Photo]:
    photos = []
    for xml_path in sorted(data_dir.rglob("*.xml")):
        relative_xml = PurePosixPath(xml_path.relative_to(data_dir).as_posix())
        if is_excluded(relative_xml):
            raise SystemExit(f"STOPPED: excluded data found on disk: {relative_xml}")
        country, split = country_and_split(relative_xml)
        if split != "train":
            continue
        if country in LEFT_OUT_COUNTRIES:
            report[f"left_out_{country}"] += 1
            continue
        photo_path = find_photo(xml_path)
        if photo_path is None:
            report["label_without_photo"] += 1
            continue
        try:
            size, boxes = read_boxes(xml_path)
            with Image.open(photo_path) as image:
                real_size = image.size
        except Exception:  # noqa: BLE001 (count every unreadable pair; never crash on outside data)
            report["unreadable"] += 1
            continue
        if size and size != real_size:
            report["label_size_differs_from_photo"] += 1  # trust the photo
        photos.append(Photo(photo_path, photo_path.relative_to(data_dir).as_posix(), country, real_size[0], real_size[1], boxes))
    return photos


# ---------------------------------------------------------------- cleaning


def clean_boxes(photo: Photo, report: Counter[str]) -> list[tuple[str, Box]]:
    """Keep only our 3 classes; clip boxes to the photo; drop boxes that are too thin."""
    kept = []
    for box in photo.boxes:
        name = CLASS_FROM_CODE.get(box.code)
        if name is None:
            report[f"ignored_code_{box.code}"] += 1
            continue
        clipped = Box(
            box.code, max(0.0, box.xmin), max(0.0, box.ymin), min(float(photo.width), box.xmax), min(float(photo.height), box.ymax)
        )
        if (clipped.xmin, clipped.ymin, clipped.xmax, clipped.ymax) != (box.xmin, box.ymin, box.xmax, box.ymax):
            report["boxes_clipped"] += 1
        if clipped.xmax - clipped.xmin < MIN_BOX_SIDE or clipped.ymax - clipped.ymin < MIN_BOX_SIDE:
            report["boxes_dropped_too_small"] += 1
            continue
        kept.append((name, clipped))
    return kept


def difference_hash(path: Path) -> np.ndarray:
    """256-bit fingerprint of how a photo looks: nearly identical photos get nearly identical bits."""
    with Image.open(path) as image:
        image.draft("L", (HASH_SIZE * 8, HASH_SIZE * 8))  # JPEG: decode at reduced size, much faster
        pixels = np.asarray(image.convert("L").resize((HASH_SIZE + 1, HASH_SIZE), Image.Resampling.LANCZOS), dtype=np.int16)
    bits = (pixels[:, 1:] > pixels[:, :-1]).flatten()
    return np.packbits(bits).view(">u8").astype(np.uint64)


def popcount(values: np.ndarray) -> np.ndarray:
    """Number of 1-bits in each uint64 (vectorised)."""
    values = values - ((values >> np.uint64(1)) & np.uint64(0x5555555555555555))
    values = (values & np.uint64(0x3333333333333333)) + ((values >> np.uint64(2)) & np.uint64(0x3333333333333333))
    values = (values + (values >> np.uint64(4))) & np.uint64(0x0F0F0F0F0F0F0F0F)
    return (values * np.uint64(0x0101010101010101)) >> np.uint64(56)


def group_near_duplicates(photos: list[Photo], chunk: int = 64) -> None:
    """Put near-duplicate photos into the same group (union-find over all pairs, in chunks)."""
    parent = list(range(len(photos)))

    def find(index: int) -> int:
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index

    hashes = np.stack([photo.dhash for photo in photos])  # type: ignore[misc]
    for start in range(0, len(photos), chunk):
        block = hashes[start : start + chunk]
        distance = popcount(block[:, None, :] ^ hashes[None, :, :]).sum(axis=2)
        rows, cols = np.nonzero(distance <= NEAR_DUPLICATE_BITS)
        for row, col in zip(rows.tolist(), cols.tolist(), strict=True):
            a, b = find(start + row), find(col)
            if a != b:
                parent[max(a, b)] = min(a, b)
    for index, photo in enumerate(photos):
        photo.group = find(index)


# ---------------------------------------------------------------- splitting


def assign_splits(photos: list[Photo], report: Counter[str]) -> dict[str, list[Photo]]:
    groups: dict[int, list[Photo]] = defaultdict(list)
    for photo in photos:
        groups[photo.group].append(photo)
    test_groups = {group for group, members in groups.items() if any(p.country == TEST_COUNTRY for p in members)}

    splits: dict[str, list[Photo]] = {"train": [], "val": [], "test": []}
    for group in test_groups:
        for photo in groups[group]:
            if photo.country == TEST_COUNTRY:
                splits["test"].append(photo)
            else:
                report["removed_near_duplicate_of_test_photo"] += 1  # would leak the test set into training

    rng = random.Random(SEED)  # noqa: S311 (repeatable data split, not security)
    by_country: dict[str, list[int]] = defaultdict(list)
    for group, members in sorted(groups.items()):
        if group not in test_groups:
            by_country[members[0].country].append(group)
    for _country, country_groups in sorted(by_country.items()):
        rng.shuffle(country_groups)
        total = sum(len(groups[group]) for group in country_groups)
        in_val = 0
        for group in country_groups:
            target = "val" if in_val < VAL_FRACTION * total else "train"
            splits[target].extend(groups[group])
            in_val += len(groups[group]) if target == "val" else 0
    for photos_in_split in splits.values():
        photos_in_split.sort(key=lambda photo: photo.relative)
    return splits


# ---------------------------------------------------------------- writing


def prepare_image(photo: Photo, images_dir: Path) -> tuple[str, int, int, float]:
    """Save the photo (shrunk if large) and return its new name, size and scale factor."""
    scale = min(1.0, MAX_SIDE / max(photo.width, photo.height))
    name = f"{photo.country}/{Path(photo.relative).stem}.jpg"
    target = images_dir / name
    target.parent.mkdir(parents=True, exist_ok=True)
    with Image.open(photo.path) as image:
        if scale < 1.0:
            image.draft("RGB", (round(photo.width * scale), round(photo.height * scale)))  # faster decode of big JPEGs
        image = image.convert("RGB")
        if scale < 1.0:
            image = image.resize((round(photo.width * scale), round(photo.height * scale)), Image.Resampling.LANCZOS)
        if (image.width, image.height) != (round(photo.width * scale), round(photo.height * scale)):
            raise SystemExit(f"STOPPED: unexpected size after resizing {photo.relative}")
        image.save(target, "JPEG", quality=JPEG_QUALITY)
        return name, image.width, image.height, scale


def write_split(split: str, photos: list[Photo], out_dir: Path, report: Counter[str]) -> dict[str, Any]:
    images, annotations = [], []
    class_counts: Counter[str] = Counter()
    started = time.monotonic()
    for image_id, photo in enumerate(photos, start=1):
        if image_id % 1000 == 0 or image_id == len(photos):
            elapsed = time.monotonic() - started
            left = elapsed / image_id * (len(photos) - image_id)
            print(
                f"   {split}: {image_id} of {len(photos)} photos saved ({elapsed / 60:.0f} min so far, about {left / 60:.0f} min left)",
                flush=True,
            )
        name, width, height, scale = prepare_image(photo, out_dir / "images")
        images.append(
            {"id": image_id, "file_name": name, "width": width, "height": height, "country": photo.country, "source": photo.relative}
        )
        for class_name, box in clean_boxes(photo, report):
            x, y = box.xmin * scale, box.ymin * scale
            w, h = (box.xmax - box.xmin) * scale, (box.ymax - box.ymin) * scale
            annotations.append(
                {
                    "id": len(annotations) + 1,
                    "image_id": image_id,
                    "category_id": CATEGORY_ID[class_name],
                    "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                    "area": round(w * h, 2),
                    "iscrowd": 0,
                    "rdd_code": box.code,
                }
            )
            class_counts[class_name] += 1
    coco = {"images": images, "annotations": annotations, "categories": CATEGORIES}
    annotation_path = out_dir / "annotations" / f"{split}.json"
    annotation_path.parent.mkdir(parents=True, exist_ok=True)
    annotation_path.write_text(json.dumps(coco, separators=(",", ":")))
    list_path = out_dir / "splits" / f"{split}.txt"
    list_path.parent.mkdir(parents=True, exist_ok=True)
    list_path.write_text("".join(f"{photo.relative}\n" for photo in photos))
    return {
        "photos": len(photos),
        "photos_without_boxes": len(images) - len({annotation["image_id"] for annotation in annotations}),
        "boxes_by_class": {category["name"]: class_counts[category["name"]] for category in CATEGORIES},
        "photos_by_country": dict(sorted(Counter(photo.country for photo in photos).items())),
        "sha256_list": sha256(list_path),
        "sha256_annotations": sha256(annotation_path),
    }


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def run(data_dir: Path, out_dir: Path) -> dict[str, Any]:
    if out_dir.exists() and any(out_dir.iterdir()):
        raise SystemExit(f"STOPPED: {out_dir} is not empty. Use a new folder so old files never mix in.")
    report: Counter[str] = Counter()
    print("1/5 Reading labels and photos")
    photos = collect_photos(data_dir, report)
    if not photos:
        raise SystemExit(f"STOPPED: no labelled photos found in {data_dir}")
    print(f"   {len(photos)} labelled photos", flush=True)

    print("2/5 Removing exact duplicate photos")
    seen: dict[str, Photo] = {}
    unique = []
    for photo in photos:
        photo.sha1 = hashlib.sha1(photo.path.read_bytes()).hexdigest()  # noqa: S324 (duplicate check, not security)
        if photo.sha1 in seen:
            report["exact_duplicates_removed"] += 1
            continue
        seen[photo.sha1] = photo
        unique.append(photo)

    print("3/5 Finding near-duplicate photos")
    for index, photo in enumerate(unique, start=1):
        photo.dhash = difference_hash(photo.path)
        if index % 5000 == 0:
            print(f"   fingerprinted {index} of {len(unique)} photos", flush=True)
    group_near_duplicates(unique)
    group_sizes = Counter(photo.group for photo in unique)
    report["near_duplicate_groups_with_2_or_more"] = sum(1 for size in group_sizes.values() if size > 1)
    report["photos_in_near_duplicate_groups"] = sum(size for size in group_sizes.values() if size > 1)

    print("4/5 Splitting (test = Czech, validation = 10% of each other country)")
    splits = assign_splits(unique, report)

    print("5/5 Writing prepared photos and annotations")
    summary = {name: write_split(name, members, out_dir, report) for name, members in splits.items()}
    manifest = {
        "split_version": SPLIT_VERSION,
        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "settings": {
            "seed": SEED,
            "test_country": TEST_COUNTRY,
            "left_out_countries": sorted(LEFT_OUT_COUNTRIES),
            "excluded": "United States (never unpacked)",
            "val_fraction": VAL_FRACTION,
            "max_side_px": MAX_SIDE,
            "min_box_side_px": MIN_BOX_SIDE,
            "near_duplicate": f"dHash {HASH_SIZE}x{HASH_SIZE}, at most {NEAR_DUPLICATE_BITS} bits different",
            "class_from_code": CLASS_FROM_CODE,
            "categories": CATEGORIES,
        },
        "splits": summary,
        "largest_near_duplicate_group": max(group_sizes.values()),
        "cleaning_report": dict(sorted(report.items())),
        "test_set_locked": True,
    }
    (out_dir / "split-manifest.json").write_text(json.dumps(manifest, indent=2))
    return manifest


def print_summary(manifest: dict[str, Any]) -> None:
    print("\nSet     Photos  No damage   Pothole  Surface cracking  Pavement deterioration")
    for name, details in manifest["splits"].items():
        boxes = details["boxes_by_class"]
        print(
            f"{name:<6} {details['photos']:>7}  {details['photos_without_boxes']:>9}  {boxes['pothole']:>8}"
            f"  {boxes['surface_cracking']:>16}  {boxes['pavement_deterioration']:>22}"
        )
    print("\nCleaning:")
    for key, value in manifest["cleaning_report"].items():
        print(f"  {key}: {value}")
    print(f"  largest near-duplicate group: {manifest['largest_near_duplicate_group']} photos")


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--data", type=Path, required=True, help="The 'data' folder made by fetch_rdd2022.py")
    parser.add_argument("--out", type=Path, required=True, help="A new, empty folder for the prepared dataset")
    args = parser.parse_args(argv)
    manifest = run(args.data, args.out)
    print_summary(manifest)
    print(f"\nSaved {args.out / 'split-manifest.json'}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
# 3. Check free disk space (need about 40 GB) and install one small package
!df -h /content | tail -1
!pip install -q "defusedxml==0.7.1" "pillow>=10" "numpy>=1.24"


In [ ]:
# 4. Download, check and unpack without the US part (about 10 minutes)
!python fetch_rdd2022.py --out /content/rdd2022


In [ ]:
# 5. Clean and split (about 20-40 minutes)
!python split_rdd2022.py --data /content/rdd2022/data --out /content/rdd2022/prepared


In [ ]:
# 6. Download the results to your computer (your browser saves p8_2_results.zip)
!cd /content/rdd2022/prepared && zip -q -r /content/p8_2_results.zip split-manifest.json splits
!du -sh /content/rdd2022/prepared
from google.colab import files

files.download('/content/p8_2_results.zip')
